In [7]:
import numpy as np
import pprint

from gaussian import gaussian_eliminate, back_substitution
from determinant import determinant 
from inverse import inverse
from rank_basis import rank_and_basis

def print_result(title, result):
    print(f"\n--- {title} ---")
    pprint.pprint(result)

In [8]:
PIVOT_EPS  = 1e-12
PIVOT_WARN = 1e-6
RESID_WARN = 1e-6

def verify_solution(A, x, b):
    """
    Kiểm chứng nghiệm x của hệ A · x = b bằng NumPy.
 
    Tính phần dư  r = b − A·x  và so sánh với NumPy's linalg.solve.
 
    Tham số
    -------
    A : list[list[float]] hoặc np.ndarray — ma trận hệ số
    x : list[float]       hoặc np.ndarray — nghiệm cần kiểm tra
    b : list[float]       hoặc np.ndarray — vector vế phải
 
    Trả về (dict)
    -------------
    {
      "residual_norm"  : float,   # ||b − A·x||₂
      "rel_error"      : float,   # ||x − x_np||₂ / ||x_np||₂  (nếu tính được)
      "x_numpy"        : np.ndarray | None,
      "passed"         : bool,    # True nếu residual_norm < RESID_WARN
      "message"        : str,
    }
    """
    A_np = np.array(A, dtype=float)
    x_np_in = np.array(x, dtype=float)
    b_np = np.array(b, dtype=float)
 
    residual      = b_np - A_np @ x_np_in
    residual_norm = float(np.linalg.norm(residual))
 
    # Thử giải bằng NumPy để so sánh
    x_numpy   = None
    rel_error = None
    try:
        x_np_ref  = np.linalg.solve(A_np, b_np)
        x_numpy   = x_np_ref
        denom     = np.linalg.norm(x_np_ref)
        rel_error = float(np.linalg.norm(x_np_in - x_np_ref) / denom) \
                    if denom > PIVOT_EPS else float(np.linalg.norm(x_np_in - x_np_ref))
    except np.linalg.LinAlgError:
        pass   # Ma trận suy biến — không so sánh được
 
    passed = residual_norm < RESID_WARN
 
    if passed:
        msg = f"OK — ||r||₂ = {residual_norm:.2e}"
        if rel_error is not None:
            msg += f", sai số tương đối so với NumPy = {rel_error:.2e}"
    else:
        msg = (
            f"CẢNH BÁO — ||r||₂ = {residual_norm:.2e} vượt ngưỡng {RESID_WARN:.0e}. "
            "Nghiệm có thể không chính xác."
        )
 
    return {
        "residual_norm" : residual_norm,
        "rel_error"     : rel_error,
        "x_numpy"       : x_numpy,
        "passed"        : passed,
        "message"       : msg,
    }

In [9]:
print("=== PHẦN 1.1: GIẢI HỆ PHƯƠNG TRÌNH TUYẾN TÍNH ===")

test_cases_gauss = [
    # Case 1: Hệ 3x3 có nghiệm duy nhất chuẩn
    {"A": [[3, 2, -1], [2, -2, 4], [-1, 0.5, -1]], "b": [1, -2, 0], "desc": "Hệ 3x3 có nghiệm duy nhất"},
    # Case 2: Cần hoán vị hàng (Pivot tại A[0][0] là 0)
    {"A": [[0, 1, 1], [1, 2, 1], [2, 7, 9]], "b": [2, 4, 18], "desc": "Cần hoán vị hàng (Partial Pivoting)"},
    # Case 3: Hệ vô nghiệm
    {"A": [[1, 2, 3], [4, 5, 6], [7, 8, 9]], "b": [1, 1, 1], "desc": "Hệ vô nghiệm (Ma trận suy biến)"},
    # Case 4: Hệ có vô số nghiệm
    {"A": [[1, 1, 1], [2, 2, 2], [3, 3, 3]], "b": [1, 2, 3], "desc": "Hệ có vô số nghiệm"},
    # Case 5: Ma trận lớn 5x5 ngẫu nhiên
    {"A": np.random.rand(5, 5).tolist(), "b": np.random.rand(5).tolist(), "desc": "Ma trận 5x5 ngẫu nhiên"}
]

import warnings
warnings.filterwarnings('ignore')

for i, tc in enumerate(test_cases_gauss):
    print(f"\nTest Case {i+1}: {tc['desc']}")
    try:
        U, c, x, swaps, pcols = gaussian_eliminate(tc['A'], tc['b'])
        if x:
            print(f"Nghiệm x: {x}")
            # Kiểm chứng với NumPy bằng hàm verify_solution bạn đã viết
            v = verify_solution(tc['A'], x, tc['b'])
            print(f"Kiểm chứng: {v['message']}")
        else:
            print("Kết quả: Hệ không có nghiệm duy nhất.")
    except Exception as e:
        print(f"Thông báo: {e}")

=== PHẦN 1.1: GIẢI HỆ PHƯƠNG TRÌNH TUYẾN TÍNH ===

Test Case 1: Hệ 3x3 có nghiệm duy nhất
Nghiệm x: [1.0, -1.9999999999999996, -1.9999999999999993]
Kiểm chứng: OK — ||r||₂ = 1.83e-15, sai số tương đối so với NumPy = 2.89e-16

Test Case 2: Cần hoán vị hàng (Partial Pivoting)
Nghiệm x: [1.0, 1.0, 1.0]
Kiểm chứng: OK — ||r||₂ = 0.00e+00, sai số tương đối so với NumPy = 1.10e-15

Test Case 3: Hệ vô nghiệm (Ma trận suy biến)
Kết quả: Hệ không có nghiệm duy nhất.

Test Case 4: Hệ có vô số nghiệm
Kết quả: Hệ không có nghiệm duy nhất.

Test Case 5: Ma trận 5x5 ngẫu nhiên
Nghiệm x: [-1.0796765864206768, -0.27067662942809756, -0.5266442863205444, 0.866474873351818, 1.351539686664692]
Kiểm chứng: OK — ||r||₂ = 2.99e-16, sai số tương đối so với NumPy = 1.12e-15


In [10]:
print("\n=== PHẦN 1.2: TÍNH ĐỊNH THỨC ===")

test_cases_det = [
    [[1, 0, 0], [0, 1, 0], [0, 0, 1]], # Ma trận đơn vị (det=1)
    [[1, 2, 3], [0, 4, 5], [0, 0, 6]], # Ma trận tam giác (det=24)
    [[2, 4, 6], [1, 2, 3], [1, 1, 1]], # Dòng 1 gấp đôi dòng 2 (det=0)
    [[0, 1], [1, 0]],                 # Hoán vị 1 lần (det=-1)
    np.random.randint(1, 10, (4, 4)).tolist() # Ma trận 4x4 ngẫu nhiên
]

for i, A in enumerate(test_cases_det):
    my_det = determinant(A)
    np_det = np.linalg.det(A)
    print(f"Test {i+1}: Cài đặt = {my_det:.4f} | NumPy = {np_det:.4f} | Khớp: {np.isclose(my_det, np_det)}")


=== PHẦN 1.2: TÍNH ĐỊNH THỨC ===
Test 1: Cài đặt = 1.0000 | NumPy = 1.0000 | Khớp: True
Test 2: Cài đặt = 24.0000 | NumPy = 24.0000 | Khớp: True
Test 3: Cài đặt = 0.0000 | NumPy = 0.0000 | Khớp: True
Test 4: Cài đặt = -1.0000 | NumPy = -1.0000 | Khớp: True
Test 5: Cài đặt = -1473.0000 | NumPy = -1473.0000 | Khớp: True


In [11]:
print("\n=== PHẦN 1.3: MA TRẬN NGHỊCH ĐẢO ===")

test_cases_inv = [
    [[4, 7], [2, 6]],                # Ma trận 2x2 khả nghịch
    [[1, 2, 3], [0, 1, 4], [5, 6, 0]], # Ma trận 3x3 khả nghịch
    [[1, 2, 3], [4, 5, 6], [7, 8, 9]], # Ma trận suy biến (Không nghịch đảo được)
    np.diag([1, 2, 3, 4]).tolist(),  # Ma trận đường chéo
    [[1, 0, 0], [0, 0, 1], [0, 1, 0]]  # Ma trận hoán vị
]

for i, A in enumerate(test_cases_inv):
    print(f"\nTest {i+1}:")
    try:
        inv_A = inverse(A)
        # Kiểm tra A * A^-1 có bằng I không
        res = np.dot(A, inv_A)
        is_identity = np.allclose(res, np.eye(len(A)))
        print(f"Trạng thái: Thành công. Kiểm tra A*inv(A)=I: {is_identity}")
    except Exception as e:
        print(f"Trạng thái: {e}")
    


=== PHẦN 1.3: MA TRẬN NGHỊCH ĐẢO ===

Test 1:
Trạng thái: Thành công. Kiểm tra A*inv(A)=I: True

Test 2:
Trạng thái: Thành công. Kiểm tra A*inv(A)=I: True

Test 3:
Trạng thái: Ma trận suy biến tại cột 2 (|pivot| = 7.77e-16) — không tồn tại nghịch đảo.

Test 4:
Trạng thái: Thành công. Kiểm tra A*inv(A)=I: True

Test 5:
Trạng thái: Thành công. Kiểm tra A*inv(A)=I: True


In [12]:
print("\n=== PHẦN 1.4: HẠNG VÀ CƠ SỞ ===")

test_cases_rank = [
    # Ma trận không vuông 3x5
    [[1, 2, 0, 2, 5], [-2, -5, 1, -1, -8], [0, -3, 3, 4, -1]], 
    # Ma trận vuông hạng đầy đủ
    [[1, 0], [0, 1]],
    # Ma trận không (hạng 0)
    [[0, 0], [0, 0]],
    # Ma trận có dòng phụ thuộc tuyến tính
    [[1, 2, 3], [2, 4, 6], [1, 0, 1]],
    # Ma trận 4x5 của ví dụ trong ex1.py
    [[1, 2, 0, 2, 5], [-2, -5, 1, -1, -8], [0, -3, 3, 4, -1], [3, 6, 0, 7, 14]]
]

for i, A in enumerate(test_cases_rank):
    res = rank_and_basis(A)
    print(f"\nTest {i+1} (Size {len(A)}x{len(A[0])}):")
    print(f" - Hạng: {res['rank']}")
    print(f" - Pivot columns: {res['pivot_cols']}")
    print(f" - Free columns: {res['free_cols']}")


=== PHẦN 1.4: HẠNG VÀ CƠ SỞ ===

Test 1 (Size 3x5):
 - Hạng: 3
 - Pivot columns: [0, 1, 3]
 - Free columns: [2, 4]

Test 2 (Size 2x2):
 - Hạng: 2
 - Pivot columns: [0, 1]
 - Free columns: []

Test 3 (Size 2x2):
 - Hạng: 0
 - Pivot columns: []
 - Free columns: [0, 1]

Test 4 (Size 3x3):
 - Hạng: 2
 - Pivot columns: [0, 1]
 - Free columns: [2]

Test 5 (Size 4x5):
 - Hạng: 4
 - Pivot columns: [0, 1, 3, 4]
 - Free columns: [2]
